# Итоговый Pipeline модели цены Airbnb

1. Данные и разбиение
2. Итоговый Pipeline
3. Сохранение в model.pkl
4. Загрузка и прогноз

## 1. Данные и разбиение

In [1]:
from pathlib import Path

import boto3
import pandas as pd
import numpy as np
from botocore.exceptions import BotoCoreError, ClientError

from secretkeys import ACCESS_KEY, KEY_ID

In [2]:
S3_BUCKET = "mlops26-course-bucket"
S3_KEY = "prepared/train.csv"
LOCAL_PATH = Path("data/train.csv")

s3 = boto3.client(
    "s3",
    endpoint_url="https://storage.yandexcloud.net",
    aws_access_key_id=KEY_ID,
    aws_secret_access_key=ACCESS_KEY,
    region_name="ru-central1",
)

In [3]:
def read_s3_csv(bucket, key):
    response = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(response["Body"], dtype={"listing_id": str})

In [4]:
def read_data(bucket, key, local_path=None):
    """Попробуйте S3, при сбое доступа или сети прочитайте локальную копию.

    1. В try вызовите read_s3_csv(bucket, key) и верните результат.
    2. Перехватите BotoCoreError, ClientError и OSError.
    3. Если local_path не передан, повторно поднимите исходную ошибку: raise.
    4. Иначе выведите тип ошибки и сообщение о переходе к локальному файлу.
    5. Прочитайте local_path через pd.read_csv.
       Верните DataFrame.

    Не используйте except Exception. Ошибки разбора CSV должны быть видны.
    Если локального файла нет, должен возникнуть FileNotFoundError.
    """
    try:
        return read_s3_csv(bucket, key)
    except (BotoCoreError, ClientError, OSError) as error:
        if local_path is None:
            raise
        print("ошибка S3:", type(error).__name__)
        print("читаем:", local_path)
        return pd.read_csv(local_path, dtype={"listing_id": str})

In [5]:
df = read_data(S3_BUCKET, S3_KEY, LOCAL_PATH)
df.shape

(5724, 13)

In [6]:
X = df.drop(columns=["listing_id", "snapshot_date", "price"])
y = df["price"]
X.shape, y.shape

((5724, 10), (5724,))

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_rest, y_train, y_rest = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_rest, y_rest, test_size=0.5, random_state=42)
len(X_train), len(X_val), len(X_test)

(3434, 1145, 1145)

In [8]:
# числовые и категориальные колонки
category_columns = ["neighbourhood_cleansed", "property_type", "room_type"]
numeric_columns = [column for column in X.columns if column not in category_columns]
numeric_columns

['latitude',
 'longitude',
 'accommodates',
 'bathrooms',
 'bedrooms',
 'beds',
 'minimum_nights']

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_pipeline = Pipeline([
    ("fill", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])

In [10]:
from sklearn.preprocessing import OneHotEncoder

categorical_pipeline = Pipeline([
    ("fill", SimpleImputer(strategy="constant", fill_value="missing")),
    ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

In [11]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
    ("numbers", numeric_pipeline, numeric_columns),
    ("category", categorical_pipeline, category_columns)
], remainder="drop")

## 2. Итоговый Pipeline
> обработка и Ridge в одной цепочке, учим только на train

In [12]:
from sklearn.linear_model import Ridge

final_pipeline = Pipeline([
    ("prepare", preprocessor),
    ("model", Ridge())
])
final_pipeline.fit(X_train, y_train)
list(final_pipeline.named_steps)

['prepare', 'model']

In [13]:
from sklearn.metrics import mean_absolute_error

round(mean_absolute_error(y_val, final_pipeline.predict(X_val)), 2)

113.18

## 3. Сохранение в model.pkl

In [14]:
import pickle

artifact_dir = Path("artifacts")
artifact_dir.mkdir(exist_ok=True)
model_path = artifact_dir / "model.pkl"

with model_path.open("wb") as file:
    pickle.dump(final_pipeline, file)

model_path, model_path.stat().st_size

(PosixPath('artifacts/model.pkl'), 5261)

## 4. Загрузка и прогноз
> Читаем model.pkl обратно и просто работаем на ней без фита

In [15]:
with model_path.open("rb") as file:
    loaded_model = pickle.load(file)

type(loaded_model).__name__, list(loaded_model.named_steps)

('Pipeline', ['prepare', 'model'])

In [16]:
# прогнозы до сохранения и после загрузки
probe = X_val.head(3)
before_save = final_pipeline.predict(probe)
after_load = loaded_model.predict(probe)
after_load.round(1), np.array_equal(before_save, after_load)

(array([186.9,  15.6, 228.4]), True)

In [17]:
# сырые строки с пропусками
rows_with_gaps = X_val[X_val.isna().any(axis=1)].head(3)
rows_with_gaps[["bathrooms", "bedrooms", "beds", "minimum_nights"]]

,bathrooms,bedrooms,beds,minimum_nights
2897,1.0,NaN,1.0,1.0
3910,NaN,1.0,1.0,1.0
3513,1.0,NaN,1.0,2.0


In [18]:
loaded_model.predict(rows_with_gaps).round(1)

array([ 32.1,  42.3, 309.7])

> модель не упала на пустых клетках, пропуски заполнились медианами из train